In [ ]:
import gc
import polars as pl

TRAIN_DIR = "data/train"
TEST_DIR = "data/test"
SAMPLE_DIR = "data/sample"

In [ ]:
def stratified_sample(target_size: int = 100_000, seed: int = 42) -> tuple[pl.DataFrame, pl.Series]:
    """
    Create a stratified sample of the base of the dataset maintaining representation across all WEEK_NUM values.

    Args:
        target_size: Target number of rows (default = 100K)
        seed: Sample seed (default = 42)

    Returns:
        A tuple of (sampled DataFrame, sampled case_ids Series)
    """

    # Load full dataset
    train_df = pl.read_parquet(f"{TRAIN_DIR}/train_base.parquet")
    total_rows = len(train_df)
    print(f"Total rows: {total_rows:,}")

    # Get week distribution
    week_counts = train_df.group_by("WEEK_NUM").agg(
        pl.len().alias("count")).sort("WEEK_NUM")
    print(f"Week distribution:\n{week_counts}")

    # Calculate sampling fraction
    sampling_fraction = target_size / total_rows
    print(f"Base sampling fraction: {sampling_fraction:.4f}")

    # Stratified sampling: sample from each week proportionally
    # This ensures every week is represented
    sampled_dfs = []

    for week_row in week_counts.iter_rows(named=True):
        week_num = week_row["WEEK_NUM"]
        week_count = week_row["count"]

        # Calculate samples for this week (proportional sampling)
        # At least 1 sample per week
        samples_for_week = max(1, int(week_count * sampling_fraction))

        # Sample from this week
        week_data = train_df.filter(pl.col("WEEK_NUM") == week_num)
        week_sample = week_data.sample(
            n=min(samples_for_week, week_count), seed=seed)
        sampled_dfs.append(week_sample)

    # Delete the original dataframe to free memory
    del train_df
    gc.collect()
    print("\nOriginal dataset removed from memory")

    # Combine all samples
    sampled_df = pl.concat(sampled_dfs)
    final_size = len(sampled_df)
    print(f"\nFinal sampled size: {final_size:,} rows")

    # Verify all weeks are represented
    sampled_week_counts = sampled_df.group_by("WEEK_NUM").agg(
        pl.len().alias("count")).sort("WEEK_NUM")
    print(
        f"Weeks in sample: {len(sampled_week_counts)} (should match {len(week_counts)})")

    # Save the sampled dataset
    output_path = f"{SAMPLE_DIR}/train_base_sampled.parquet"
    sampled_df.write_parquet(output_path)
    print(f"Saved sampled base to {output_path}")

    # Extract and return case_ids for filtering other files
    sampled_case_ids = sampled_df.select("case_id").to_series()

    return sampled_df, sampled_case_ids


def sample_all_files(sampled_case_ids: pl.Series, seed: int = 42) -> None:
    """
    Sample all parquet files in TRAIN_DIR to include only cases in sampled_case_ids.
    Handles files with multiple rows per case (depth = 1, 2).

    Args:
        sampled_case_ids: Series of case_ids from the stratified sample
        seed: Random seed for reproducibility
    """
    import os

    # Get unique case_ids as a set for efficient lookup
    case_id_set = set(sampled_case_ids.to_list())

    # List all parquet files in TRAIN_DIR
    train_files = [f for f in os.listdir(TRAIN_DIR) if f.endswith(
        '.parquet') and f != 'train_base.parquet']

    print(f"\nFound {len(train_files)} additional parquet files to sample")

    for file in train_files:
        file_path = f"{TRAIN_DIR}/{file}"
        print(f"\nProcessing {file}...")

        try:
            # Read the file
            df = pl.read_parquet(file_path)
            original_size = len(df)

            # Filter to only include sampled case_ids
            df_sampled = df.filter(pl.col("case_id").is_in(case_id_set))
            sampled_size = len(df_sampled)

            # Save the sampled file
            output_name = file.replace('.parquet', '_sampled.parquet')
            output_path = f"{SAMPLE_DIR}/{output_name}"
            df_sampled.write_parquet(output_path)

            print(
                f"  Original: {original_size:,} rows → Sampled: {sampled_size:,} rows")
            print(f"  Saved to {output_path}")

            del df, df_sampled
            gc.collect()

        except Exception as e:
            print(f"  ERROR: {e}")

    print("\nAll files sampled successfully!")

In [ ]:
# Execute the sampling on all files
train_base_sample, sampled_case_ids = stratified_sample()
print(f"\nTotal sampled case_ids: {len(sampled_case_ids):,}")

# Sample all other parquet files based on the sampled case_ids
sample_all_files(sampled_case_ids)

# Clean up
del train_base_sample
del sampled_case_ids
gc.collect()
print("\nSampling complete!")